In [ ]:
import re
import pandas as pd
from pathlib import Path

In [ ]:
current_directory= Path.cwd()
project_root=current_directory.parent
actual_file=project_root/"data"/"raw"/"datos_sucios_hito1.csv"

if not actual_file.exists():
    raise FileNotFoundError("Falta el insumo de trabajo. Debe nombrarlo como \"datos_sucios_hito1.csv\" y guardarlo en data/raw/")
df_namedates=pd.read_csv(actual_file)
df_namedates

In [ ]:
names_nodata=df_namedates["Nombre_Informante"].isna()|(df_namedates["Nombre_Informante"].str.replace(" ","")=="")|(df_namedates["Nombre_Informante"].str.contains("N/A",case=False))|(df_namedates["Nombre_Informante"].str.contains("No registra",case=False))
df_namedates.loc[names_nodata,"Nombre_Informante"]="Sin dato"

names_data=df_namedates["Nombre_Informante"]!="Sin dato"
df_namedates.loc[names_data,"Nombre_Informante"]=df_namedates.loc[names_data,"Nombre_Informante"].str.title()
df_namedates


In [ ]:
months_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

def numeric_month(match):
    '''
    Convierte el mes textual a su número correspondiente y reorganiza los elementos de la fecha para que coincida con el formato día/mes/año

    Args:
        match (re.Match): objeto de coincidencia entregado automáticamente por .str.replace(), del cual se extraen los grupos nombrados capturados por el patrón.
    
    Returns:
        str: cadena de texto con la fecha normalizada para que coincida con el formato día/mes/año.
    '''
    text_month=match.group("month")
    return f"{match.group("day")}/{months_map[text_month]}/{match.group("year")}"

df_namedates["Fecha_Registro"]=df_namedates["Fecha_Registro"].str.replace(r"(?P<month>\w{3})\s*(?P<day>\d{2}),\s*(?P<year>\d{2,4})",numeric_month,regex=True)
df_namedates[["Fecha_Registro"]]

In [ ]:
def month_year_date(match):
    '''
    Reorganiza los elementos de la fecha para que coincidan con el formato día/mes/año

    Args:
        match (re.Match): objeto de coincidencia entregado automáticamente por .str.replace(), del cual se extraen los grupos nombrados capturados por el patrón.
    
    Returns:
        str: cadena de texto con la fecha normalizada para que coincida con el formato día/mes/año
    
    Notes:
        Para resolver los casos ambiguos entre día/mes (o incluso año cuando solo tiene dos dígitos) en posición inicial, la función asume que el primer elemento siempre es un 
        día, pues el formato común en el contexto de creación y uso de este DataFrame es día/mes/año.
    '''
    if len(match.group("a"))==2: 
        return f"{match.group("a")}/{match.group("b")}/{match.group("c")}"
    else:
        return f"{match.group("c")}/{match.group("b")}/{match.group("a")}"

df_namedates["Fecha_Registro"]=df_namedates["Fecha_Registro"].str.replace(r"(?P<a>\d{2,4})[-/](?P<b>\d{2})[-/](?P<c>\d{2,4})",month_year_date,regex=True)
df_namedates